<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/SQLite%20Telemetry%20Database%20Validation%20and%20Consistency%20Engine%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
"""
Group 11: SQLite Telemetry Database Validation and Consistency Engine
CSC 806 - Research Methods in Computer Science
Advisor: Dr. Ronald Ojino

This script validates the structural consistency, physical bounds, data integrity,
and formulaic accuracy of the cloud telemetry database before public archiving.
If no database is present, it runs in "Demo Mode" by creating a test database
with seeded data anomalies to demonstrate its extensive diagnostic capabilities.
"""

import os
import sys
import sqlite3
import json
import math
from datetime import datetime, timezone

# Expected schema definition
EXPECTED_COLUMNS = {
    "id": "INTEGER",
    "timestamp": "TEXT",
    "instance_name": "TEXT",
    "region": "TEXT",
    "cpu_utilization": "REAL",
    "ram_utilization": "REAL",
    "power_draw": "REAL",
    "carbon_intensity": "REAL",
    "pue": "REAL",
    "hourly_cost": "REAL",
    "operational_carbon": "REAL",
    "efficiency_ratio": "REAL"
}

def create_demo_database(db_path):
    """Generates a demo SQLite database loaded with valid and deliberate invalid records."""
    print(f"[*] Demo Mode: Creating simulated database at '{db_path}'...")
    # Ensure the directory exists
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Create schema
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS telemetry (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT,
        instance_name TEXT NOT NULL,
        region TEXT NOT NULL,
        cpu_utilization REAL,
        ram_utilization REAL,
        power_draw REAL,
        carbon_intensity REAL,
        pue REAL,
        hourly_cost REAL,
        operational_carbon REAL,
        efficiency_ratio REAL
    );
    """)

    # 1. Seed 15 perfectly valid entries
    valid_entries = []
    regions = [("US-West (Oregon)", 79.0, 1.12, 0.08),
               ("US-East (Virginia)", 379.0, 1.15, 0.06),
               ("EU-Central (Frankfurt)", 311.0, 1.18, 0.13),
               ("AP-South (South Africa)", 840.0, 1.25, 0.07)]

    for i in range(1, 16):
        reg_name, ci, pue, cost = regions[i % len(regions)]
        cpu = 30.0 + (i * 4.5) % 65.0
        ram = 40.0 + (i * 3.2) % 55.0
        # Linear power model: P(u) = 105.0 + (245.0 - 105.0) * (u / 100.0)
        power = 105.0 + (140.0 * (cpu / 100.0))
        # E_CO2 = (Power/1000) * PUE * CI
        op_carbon = (power / 1000.0) * pue * ci
        eff = cpu / power

        valid_entries.append((
            f"2026-08-15T12:{i:02d}:00Z",
            f"vm-prod-node-{i:02d}",
            reg_name,
            cpu,
            ram,
            power,
            ci,
            pue,
            cost,
            op_carbon,
            eff
        ))

    cursor.executemany("""
    INSERT INTO telemetry (timestamp, instance_name, region, cpu_utilization, ram_utilization, power_draw, carbon_intensity, pue, hourly_cost, operational_carbon, efficiency_ratio)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, valid_entries)

    # 2. Seed Deliberate Anomalies for validation demo
    anomalies = [
        # Anomaly 1: Negative CPU Utilization
        ("2026-08-15T13:01:00Z", "vm-anomaly-neg-cpu", "US-West (Oregon)", -15.5, 50.0, 120.0, 79.0, 1.12, 0.08, 13.44, 0.15),
        # Anomaly 2: Extreme unphysical PUE (> 2.0)
        ("2026-08-15T13:02:00Z", "vm-anomaly-extreme-pue", "US-East (Virginia)", 65.0, 60.0, 196.0, 379.0, 3.50, 0.06, 259.99, 0.33),
        # Anomaly 3: Null value in critical continuous fields
        ("2026-08-15T13:03:00Z", "vm-anomaly-missing-power", "EU-Central (Frankfurt)", 45.0, 40.0, None, 311.0, 1.18, 0.13, 0.0, 0.0),
        # Anomaly 4: Formulaic Drift (Operational carbon incorrectly pre-calculated)
        ("2026-08-15T13:04:00Z", "vm-anomaly-carbon-drift", "AP-South (South Africa)", 50.0, 50.0, 175.0, 840.0, 1.25, 0.07, 9999.99, 0.285)
    ]

    cursor.executemany("""
    INSERT INTO telemetry (timestamp, instance_name, region, cpu_utilization, ram_utilization, power_draw, carbon_intensity, pue, hourly_cost, operational_carbon, efficiency_ratio)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """, anomalies)

    conn.commit()
    conn.close()
    print(f"[+] Demo database populated successfully with {len(valid_entries)} valid records and {len(anomalies)} intentional anomaly cases.\n")

def run_validation(db_path):
    """Runs a complete structural, integrity, formulaic, and statistical validation suite."""
    report = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "database_file": os.path.basename(db_path),
        "sqlite_integrity_check": "FAILED",
        "schema_validation": "FAILED",
        "record_count": 0,
        "anomalies_detected": [],
        "statistical_outliers": [],
        "overall_status": "FAILED"
    }

    print("="*80)
    print(f"      DATABASE CONSISTENCY & EMPIRICAL ARCHIVING VALIDATION REPORT")
    print("="*80)
    print(f"Target Database File: {db_path}")
    print(f"Validation Timestamp: {report['timestamp']}")
    print("-"*80)

    # 1. Establish connection
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
    except Exception as e:
        print(f"❌ Critical Connection Error: Could not connect to SQLite database. {e}")
        report["overall_status"] = "ERROR: Cannot Connect"
        return report

    # 2. SQLite Low-Level Integrity Check
    print("[*] Phase 1: Checking low-level SQLite database file integrity...")
    try:
        cursor.execute("PRAGMA integrity_check;")
        res = cursor.fetchone()[0]
        if res.lower() == "ok":
            print("  ✅ PRAGMA integrity_check: OK (No physical file corruption detected)")
            report["sqlite_integrity_check"] = "PASSED"
        else:
            print(f"  ❌ PRAGMA integrity_check: CORRUPTED ({res})")
            report["anomalies_detected"].append({"type": "physical_corruption", "detail": res})
    except Exception as e:
         print(f"  ❌ Error executing integrity check: {e}")

    # 3. Schema Validation
    print("\n[*] Phase 2: Validating table schemas and constraints...")
    try:
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='telemetry';")
        table_exists = cursor.fetchone()
        if not table_exists:
            print("  ❌ Schema Error: Table 'telemetry' is missing from the database!")
            report["schema_validation"] = "FAILED (Missing Table)"
            conn.close()
            return report

        # Check columns
        cursor.execute("PRAGMA table_info(telemetry);")
        columns = cursor.fetchall()
        db_cols = {col[1]: col[2].upper() for col in columns}

        schema_errors = 0
        for expected_col, expected_type in EXPECTED_COLUMNS.items():
            if expected_col not in db_cols:
                print(f"  ❌ Schema Error: Expected column '{expected_col}' is missing.")
                schema_errors += 1
            else:
                obs_type = db_cols[expected_col]
                # Allow minor sqlite3 types variations (e.g., INT/INTEGER, FLOAT/REAL)
                type_ok = (expected_type == obs_type) or \
                          (expected_type == "REAL" and obs_type in ["FLOAT", "DOUBLE", "REAL"]) or \
                          (expected_type == "INTEGER" and obs_type in ["INT", "INTEGER"])
                if not type_ok:
                    print(f"  ⚠️ Schema Type Warning: Column '{expected_col}' expects {expected_type} but observed {obs_type}.")

        if schema_errors == 0:
            print("  ✅ Schema Structure: PASSED (All required columns are present with valid types)")
            report["schema_validation"] = "PASSED"
        else:
            report["schema_validation"] = f"FAILED ({schema_errors} structural errors)"
    except Exception as e:
        print(f"  ❌ Schema processing error: {e}")
        report["schema_validation"] = "ERROR"

    # 4. Record Count and Row Ingestion
    cursor.execute("SELECT COUNT(*) FROM telemetry;")
    total_records = cursor.fetchone()[0]
    report["record_count"] = total_records
    print(f"\n[*] Database Size: {total_records} empirical records found.")

    if total_records == 0:
        print("  ⚠️ Warning: Database contains 0 records. Skipping content validations.")
        conn.close()
        return report

    # Fetch all records
    cursor.execute("SELECT * FROM telemetry;")
    columns_names = [description[0] for description in cursor.description]
    records = [dict(zip(columns_names, row)) for row in cursor.fetchall()]

    # 5. Logical and Physical Bound Validations
    print("\n[*] Phase 3: Executing multi-criteria physical bounds & formula validation...")

    anomalies = []

    # Track statistics for anomaly z-scores
    cpu_vals = []
    ram_vals = []
    power_vals = []

    for r in records:
        rid = r["id"]
        instance = r["instance_name"]

        # Keep track of numerical records for outlier statistics
        if r["cpu_utilization"] is not None: cpu_vals.append(r["cpu_utilization"])
        if r["ram_utilization"] is not None: ram_vals.append(r["ram_utilization"])
        if r["power_draw"] is not None: power_vals.append(r["power_draw"])

        # Null constraint check
        critical_fields = ["timestamp", "instance_name", "region", "cpu_utilization", "power_draw", "carbon_intensity", "pue"]
        for field in critical_fields:
            if r[field] is None:
                anomalies.append({
                    "id": rid,
                    "instance": instance,
                    "field": field,
                    "error_type": "NULL_VIOLATION",
                    "details": f"Critical field '{field}' contains a null value, violating archiving completeness."
                })
                continue

        # If any value is null, skip bounds check to avoid crash
        if any(r[f] is None for f in critical_fields):
            continue

        # Physical boundary checks
        cpu = r["cpu_utilization"]
        ram = r["ram_utilization"]
        power = r["power_draw"]
        ci = r["carbon_intensity"]
        pue = r["pue"]
        cost = r["hourly_cost"]
        op_carbon = r["operational_carbon"]
        eff_ratio = r["efficiency_ratio"]

        # CPU range: [0, 100]
        if cpu < 0.0 or cpu > 100.0:
            anomalies.append({
                "id": rid, "instance": instance, "field": "cpu_utilization",
                "error_type": "OUT_OF_BOUNDS", "details": f"Value {cpu}% is outside the physical range of [0.0%, 100.0%]."
            })

        # RAM range: [0, 100]
        if ram is not None and (ram < 0.0 or ram > 100.0):
            anomalies.append({
                "id": rid, "instance": instance, "field": "ram_utilization",
                "error_type": "OUT_OF_BOUNDS", "details": f"Value {ram}% is outside the physical range of [0.0%, 100.0%]."
            })

        # Power bounds: strict positive
        if power <= 0.0:
            anomalies.append({
                "id": rid, "instance": instance, "field": "power_draw",
                "error_type": "OUT_OF_BOUNDS", "details": f"Power draw {power}W must be strictly greater than 0."
            })

        # Carbon intensity: strict positive
        if ci <= 0.0:
            anomalies.append({
                "id": rid, "instance": instance, "field": "carbon_intensity",
                "error_type": "OUT_OF_BOUNDS", "details": f"Grid carbon intensity {ci} gCO2e/kWh must be positive."
            })

        # PUE bounds: [1.0, 2.0]
        if pue < 1.0 or pue > 2.0:
            anomalies.append({
                "id": rid, "instance": instance, "field": "pue",
                "error_type": "OUT_OF_BOUNDS", "details": f"PUE value {pue} is outside the standard datacenter operational limits [1.0, 2.0]."
            })

        # Cost bounds: positive
        if cost is not None and cost < 0.0:
            anomalies.append({
                "id": rid, "instance": instance, "field": "hourly_cost",
                "error_type": "OUT_OF_BOUNDS", "details": f"Hourly VM cost ${cost} cannot be negative."
            })

        # 6. Formula Accuracy and Dynamic Consistency
        # Operational Carbon Formula: E = (Power / 1000) * PUE * Carbon_Intensity
        expected_carbon = (power / 1000.0) * pue * ci
        carbon_drift = abs(op_carbon - expected_carbon)
        if carbon_drift > 1e-3:
            anomalies.append({
                "id": rid, "instance": instance, "field": "operational_carbon",
                "error_type": "FORMULA_DRIFT",
                "details": f"Calculated carbon {op_carbon} gCO2e does not match formula expected {expected_carbon:.4f} (Drift: {carbon_drift:.4f} g)."
            })

        # Efficiency ratio: η = CPU / Power
        expected_eff = cpu / power
        eff_drift = abs(eff_ratio - expected_eff)
        if eff_drift > 1e-4:
            anomalies.append({
                "id": rid, "instance": instance, "field": "efficiency_ratio",
                "error_type": "FORMULA_DRIFT",
                "details": f"Efficiency ratio {eff_ratio} does not match expected CPU/W ratio of {expected_eff:.6f} (Drift: {eff_drift:.6f})."
            })

    # Print anomalies
    if anomalies:
        print(f"  ❌ Data Consistency: FAILED ({len(anomalies)} integrity exceptions flagged)")
        for anomaly in anomalies:
            print(f"    - Row {anomaly['id']} ({anomaly['instance']}): [{anomaly['error_type']}] {anomaly['details']}")
    else:
        print("  ✅ Data Consistency: PASSED (All record boundaries and calculations are logically coherent)")

    report["anomalies_detected"] = anomalies

    # 7. Statistical Outlier Detection (Z-Score method)
    print("\n[*] Phase 4: Scanning for statistical outliers (z-score >= 3.0)...")
    outliers = []

    def find_outliers_zscore(values, field_name):
        if len(values) < 5:
            return []
        mean = sum(values) / len(values)
        variance = sum((x - mean) ** 2 for x in values) / len(values)
        std_dev = math.sqrt(variance)

        field_outliers = []
        if std_dev == 0:
            return []

        for r in records:
            val = r[field_name]
            if val is None:
                continue
            z_score = abs(val - mean) / std_dev
            if z_score >= 3.0:
                field_outliers.append({
                    "id": r["id"],
                    "instance": r["instance_name"],
                    "field": field_name,
                    "value": val,
                    "z_score": z_score,
                    "error_type": "STATISTICAL_OUTLIER",
                    "details": f"Value {val} is a major statistical anomaly (z-score = {z_score:.2f} > 3.0)."
                })
        return field_outliers

    cpu_outliers = find_outliers_zscore(cpu_vals, "cpu_utilization")
    ram_outliers = find_outliers_zscore(ram_vals, "ram_utilization")
    power_outliers = find_outliers_zscore(power_vals, "power_draw")

    outliers.extend(cpu_outliers + ram_outliers + power_outliers)
    report["statistical_outliers"] = outliers

    if outliers:
        print(f"  ⚠️ Warning: Detected {len(outliers)} statistical outlier records (Z >= 3.0):")
        for outlier in outliers:
            print(f"    - Row {outlier['id']} ({outlier['instance']}): {outlier['details']}")
    else:
        print("  ✅ Statistical Outlier Scan: PASSED (No extreme outlier values detected)")

    # 8. Final Decision Logic
    is_ok = (report["sqlite_integrity_check"] == "PASSED" and
             report["schema_validation"] == "PASSED" and
             len(anomalies) == 0)

    report["overall_status"] = "PASSED (Ready for Archive)" if is_ok else "FAILED (Needs Remediation)"

    print("\n" + "="*80)
    print(f"      FINAL AUDIT RESULT: {report['overall_status'].upper()}")
    print("="*80)
    if is_ok:
        print("🎉 Congratulations! Your empirical trace dataset has achieved 100% scientific consistency.")
        print("The files are structured, validated, logically balanced, and ready for public archiving.")
    else:
        print("❌ Warning: Validation issues have been detected in the database file.")
        print("Please run database cleaning scripts to fix physical bounds or formulaic drift issues.")
    print("="*80)

    conn.close()
    return report

def main():
    # Detect target database or activate demo mode
    # Default to a temporary path that is guaranteed to be writable in Colab
    db_path = "/tmp/cloud_telemetry.db"

    # If a command-line argument is passed, try to use it as the database path.
    # Note: In environments like Colab, sys.argv often contains '-f' as the first argument,
    # which can interfere if not handled carefully. For this fix, we'll ensure
    # the directory exists for whatever path is chosen.
    if len(sys.argv) > 1 and sys.argv[1] != '-f': # Added check for '-f'
        db_path = sys.argv[1]

    # Check if database exists
    if not os.path.exists(db_path):
        print(f"[*] Warning: Target SQLite database not found at standard path '{db_path}'.")
        # In this sandbox environment, we fall back to generating a demo database
        db_path = "/tmp/demo_telemetry.db" # Changed to /tmp
        if not os.path.exists(db_path):
            create_demo_database(db_path)

    # Run validation suite
    report_data = run_validation(db_path)

    # Write JSON report to workspace
    report_out_path = "/tmp/db_validation_report.json" # Changed to /tmp
    with open(report_out_path, "w") as f:
        json.dump(report_data, f, indent=4)
    print(f"\n[*] JSON-formatted machine-readable diagnostic report written to: {report_out_path}")

if __name__ == "__main__":
    main()

[*] Warning: Target SQLite database not found at standard path '/tmp/cloud_telemetry.db'.
[*] Demo Mode: Creating simulated database at '/tmp/demo_telemetry.db'...
[+] Demo database populated successfully with 15 valid records and 4 intentional anomaly cases.

      DATABASE CONSISTENCY & EMPIRICAL ARCHIVING VALIDATION REPORT
Target Database File: /tmp/demo_telemetry.db
Validation Timestamp: 2026-08-15T21:45:32.311186+00:00
--------------------------------------------------------------------------------
[*] Phase 1: Checking low-level SQLite database file integrity...
  ✅ PRAGMA integrity_check: OK (No physical file corruption detected)

[*] Phase 2: Validating table schemas and constraints...
  ✅ Schema Structure: PASSED (All required columns are present with valid types)

[*] Database Size: 19 empirical records found.

[*] Phase 3: Executing multi-criteria physical bounds & formula validation...
  ❌ Data Consistency: FAILED (9 integrity exceptions flagged)
    - Row 16 (vm-anomaly-ne